# EMR Summarizer: one record, end to end

This notebook walks through the pipeline for **one** MTSamples report. Run it from the repo root after
the pipeline has run at least once (it reuses cached embeddings and model outputs, so it costs nothing).

Steps: raw report → anonymized bundle → chunks → vector search → model summary → grounding check.

In [ ]:
import json, os
os.chdir("..") if os.path.basename(os.getcwd()) == "notebooks" else None
from emr.common import read_json, path_for
from emr.config import load_config
config = load_config()
record_id = read_json(config["paths"]["test_set"])[0]
record_id

## 1. Raw vs anonymized text
The converter turned the CSV row into an **EMRBundle**. The anonymizer replaced names, dates, places and IDs
with tags such as `<PERSON>` and `<DATE>`; the patient is known only by a random UUID.

In [ ]:
raw = read_json(path_for(config["paths"]["bundles_raw"], record_id))
anon = read_json(path_for(config["paths"]["bundles"], record_id))
print("patient_id:", anon["patient"]["patient_id"])
print("department:", anon["encounter"]["department"], "| doc_type:", anon["documents"][0]["doc_type"])
print("\n--- anonymized text (first 800 chars) ---\n", anon["documents"][0]["text"][:800])

## 2. Chunks
The report is split at its ALL-CAPS section headings. Each chunk stores exact character offsets, so
`text[start:end] == chunk text` always holds.

In [ ]:
from emr.chunking import chunk_bundle
chunks = chunk_bundle(anon)
text = anon["documents"][0]["text"]
for c in chunks:
    assert text[c["start"]:c["end"]] == c["text"]
    print(f'{c["chunk_id"]:28s} {c["section"][:45]:45s} chars {c["start"]}-{c["end"]}')

## 3. Vector search (the "R" in RAG)
Each chunk was embedded with `gemini-embedding-001` and stored in ChromaDB. A question is embedded the same way
and compared with the chunks of **this record only** (filter on `record_id`). Lower distance = more similar.

In [ ]:
from emr import store
from emr.llm.base import make_llm
col = store.collection()
embedder = make_llm(config["embedding"])
question = "What is the plan and follow-up?"
for hit in store.query(col, embedder.embed([question], task="query")[0], record_id, k=3):
    print(f'{hit["distance"]:.3f}  [{hit["section"][:40]}]  {" ".join(hit["text"].split())[:100]}...')

## 4. The prompt every model receives
Retrieval output is cached per record, so every model gets exactly the same chunks and the same instructions.

In [ ]:
from emr.summarize import build_prompt, SYSTEM_PROMPT
context = read_json(path_for(config["paths"]["cache"] + "/context", record_id))
print(SYSTEM_PROMPT[:600], "...\n")
print(build_prompt(context)[:1200], "...")

## 5. The model's summary and its evidence
The model quoted the source for every claim; `grounding.py` found each quote and replaced it with
character offsets. Below, each problem is printed with the exact source text its evidence points to.

In [ ]:
model = "gemini-flash-lite"
summary = read_json(path_for(f"results/{model}", record_id))
report = read_json(path_for(f"results/{model}/_reports", record_id))
print("ONE-LINER:", summary["one_liner"], "\n")
for p in summary["problems"]:
    quotes = [text[e["start"]:e["end"]] for e in p["evidence"]]
    print(f'- {p["name"]} ({p["status"]})  <- {quotes}')
print("\nAction items:")
for a in summary["action_items"]:
    print(f'- [{a["priority"]}] {a["action"]}')

## 6. Checks
Grounding statistics and full JSON-Schema validation (same checks as `tools/validate.py`).

In [ ]:
from emr.common import summary_errors
g = report["grounding"]
print(f'claims grounded: {g["supported_claims"]}/{g["claims"]}  | unsupported: {g["unsupported"]}')
print("schema errors:", summary_errors(anon, summary) or "none")